# Runs explorer

A quick look at the **`runs`** provenance table (M4 persistence layer). One row per agent run: model, config, tokens, convergence, and references to the emitted TAF (`tafs`), the worksheet (`taf_worksheets`), and the frozen transcript blob on disk.

This notebook opens a **demo** DB and populates one synthetic run if it is empty, so it always has something to show. To look at real data later, point `DB` at `data/forecaster.duckdb` (once `collect.py` writes to it).

In [ ]:
import json
import sys
from pathlib import Path

# Find the repo root (has pyproject.toml) so scripts/ is importable and paths resolve.
_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / "scripts"))

import populate_runs_demo as demo
from forecaster import store

DB = demo.DEMO_DB   # swap for str(_root / 'data' / 'forecaster.duckdb') to see real runs

# Populate one synthetic run if the demo table is missing or empty (idempotent).
try:
    _c = store.connect(DB, read_only=True)
    _n = _c.execute("SELECT count(*) FROM runs").fetchone()[0]
    _c.close()
except Exception:
    _n = 0
if not _n:
    demo.populate()
    print("populated demo DB with one example run")

con = store.connect(DB, read_only=True)
print("DB:", DB)
print("runs:", con.execute("SELECT count(*) FROM runs").fetchone()[0],
      "| tafs:", con.execute("SELECT count(*) FROM tafs").fetchone()[0],
      "| evaluations:", con.execute("SELECT count(*) FROM evaluations").fetchone()[0])

## The runs table (curated columns)

In [ ]:
con.execute("""
    SELECT run_id, station, model, worksheet_mode, stop_reason, convergence,
           n_steps, n_tool_calls, prompt_tokens, completion_tokens,
           taf_clean, taf_id, worksheet_id
    FROM runs
    ORDER BY created_at DESC
""").df()

## One run in full (transposed)

All columns for the most recent run, read top-to-bottom. `transcript_path` points at the frozen `messages.json`; `taf_id` / `worksheet_id` join to the archived TAF and worksheet.

In [ ]:
con.execute("SELECT * FROM runs ORDER BY created_at DESC LIMIT 1").df().T

## Runs joined to the TAF they emitted

`runs.taf_id -> tafs.taf_id`. A no-emit / fatal run has `taf_id = NULL` and shows no TAF text here.

In [ ]:
con.execute("""
    SELECT r.run_id, r.model, r.convergence,
           r.prompt_tokens + r.completion_tokens AS tokens,
           t.producer_kind, t.valid_from_utc, t.valid_to_utc, t.raw_taf
    FROM runs r
    LEFT JOIN tafs t ON r.taf_id = t.taf_id
    ORDER BY r.created_at DESC
""").df()

## Pending evaluations

Each emitted TAF gets a `pending` evaluation at collection time. `score_taf.py --pending` (M4 step 4) fills in the scores once the validity window elapses and observations accumulate.

In [ ]:
con.execute("SELECT evaluation_id, station, valid_from, valid_to, status FROM evaluations").df()

## Peek at a transcript blob

The full frozen `messages` array (prompt + reasoning + every tool call and the weather data it returned + the base64 images the model saw) lives as a file referenced by `transcript_path`.

In [ ]:
tp = con.execute("SELECT transcript_path FROM runs ORDER BY created_at DESC LIMIT 1").fetchone()[0]
msgs = json.loads(Path(tp).read_text())
print(tp)
print("messages:", len(msgs), "| roles:", [m["role"] for m in msgs])
for m in msgs:
    if isinstance(m["content"], list):
        kinds = [p.get("type") for p in m["content"]]
        print(f"  {m['role']:9} -> multimodal parts: {kinds}")
    else:
        text = (m.get("content") or "").replace("\n", " ")
        tag = " +tool_calls" if m.get("tool_calls") else ""
        print(f"  {m['role']:9}{tag}: {text[:70]}")

---
_Demo data. Real runs are written by `runlog.persist_run` (via the future `collect.py`) into `data/forecaster.duckdb`; point `DB` there to explore them._